# META-CXR Training on Kaggle (2x T4 GPU)

This notebook trains the META-CXR model on the MIMIC-CXR-JPG dataset using 2x T4 GPUs via PyTorch DistributedDataParallel.

**Prerequisites:**
- Kaggle accelerator set to **GPU T4 x2**
- Two datasets attached as Kaggle input:
  - **`mimic-cxr-jpg-lite`** — JPG images + metadata CSVs (images at `p10/...`)
  - **`mimic-cxr-reported`** — radiology `.txt` reports (at `files/p10.../`)
- Internet access enabled

**Steps:** Run cells 1→6 in order.

## Cell 1 — Install Dependencies

In [ ]:
import subprocess, sys

# Kaggle has PyTorch, torchvision, numpy, pandas, scikit-learn preinstalled.
# Install only the packages that are missing.
packages = [
    "omegaconf==2.3.0",
    "pycocoevalcap",
    "scikit-image",           # latest stable; io/transform APIs are unchanged
    "torchinfo",
    "wandb",
    "loralib==0.1.1",
    "iterative-stratification",
    "iopath",
    "hi-ml-multimodal",       # provides health_multimodal used by biovil_t
    "timm==0.6.13",           # 0.6.x keeps timm.models.hub API used by dist_utils.py; PyTorch 2.x compatible
    "spacy",                  # latest 3.x; stable spacy.load() API
    "nltk==3.8.1",
    "google-cloud-storage",
    "transformers==4.44.2",   # pin for Qformer.py compatibility (apply_chunking_to_forward et al.)
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + packages,
    check=True
)

# Install peft at the specific commit used by the project
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "git+https://github.com/huggingface/peft.git@e536616888d51b453ed354a6f1e243fecb02ea08"],
    check=True
)

# Download NLTK data required by the METEOR scorer
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Download spacy English model required by blip2.py (spacy.load("en_core_web_sm"))
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)

# Detect Java installation (needed for METEOR/ROUGE scoring)
result = subprocess.run(
    "readlink -f $(which java) | sed 's|/bin/java||'",
    shell=True, capture_output=True, text=True
)
JAVA_HOME_DETECTED = result.stdout.strip()
print(f"Detected JAVA_HOME: {JAVA_HOME_DETECTED}")

# Verify GPU count
import torch
print(f"GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

## Cell 2 — Weights & Biases Setup

Đăng nhập wandb bằng Kaggle Secret `WANDB_API_KEY`.

**Cách thêm secret trên Kaggle:**  
Notebook Settings → Add-ons → Secrets → **Name:** `WANDB_API_KEY` → **Value:** API key của bạn.

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    print("wandb: API key loaded from Kaggle Secrets")
except Exception as e:
    print(f"wandb: Could not load from Kaggle Secrets ({e}) — using pre-configured key if available")

import wandb
wandb.login()
print("wandb: Logged in successfully")

## Cell 3 — Clone GitHub Repository

In [ ]:
import os

REPO_DIR = "/kaggle/working/META-CXR"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/minhphuong150505/Meta-CXR-Kaggle.git {REPO_DIR}
else:
    print(f"Repository already exists at {REPO_DIR}, pulling latest changes...")
    !git -C {REPO_DIR} pull

# Change working directory to repo root
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!ls -la

## Cell 4 — Verify Kaggle Input Datasets & Build Reports CSV

Two datasets must be attached to this notebook:

| Dataset | Slug | Contents |
|---------|------|----------|
| MIMIC-CXR-JPG-LITE | `mimic-cxr-jpg-lite` | JPG images at `p10/…` + metadata CSVs |
| mimic-cxr-reported | `mimic-cxr-reported` | `.txt` reports at `files/p10/…` |

Expected paths:
```
/kaggle/input/mimic-cxr-jpg-lite/
├── p10/p10000032/s50414267/*.jpg   ← JPG images
├── mimic-cxr-2.0.0-split.csv
├── mimic-cxr-2.0.0-chexpert.csv
└── mimic-cxr-2.0.0-metadata.csv

/kaggle/input/mimic-cxr-reported/
└── files/p10/p10000032/s50414267.txt   ← radiology reports
```

In [ ]:
import os
import glob
import re
import pandas as pd
from multiprocessing import Pool, cpu_count

IMAGES_SLUG  = "mimic-cxr-jpg-lite"
REPORTS_SLUG = "mimic-cxr-reported"

def find_mount(slug):
    direct = f"/kaggle/input/{slug}"
    if os.path.isdir(direct):
        return direct
    candidates = glob.glob(f"/kaggle/input/**/{slug}", recursive=True)
    return candidates[0] if candidates else None

# ── Images + metadata CSVs ───────────────────────────────────────────────────
KAGGLE_INPUT = find_mount(IMAGES_SLUG)
if not KAGGLE_INPUT:
    raise FileNotFoundError(
        f"Dataset '{IMAGES_SLUG}' not attached. Add it via Kaggle: Add Data → Datasets."
    )
os.environ["KAGGLE_INPUT"] = KAGGLE_INPUT
print(f"KAGGLE_INPUT (images + CSVs): {KAGGLE_INPUT}")

IMAGE_ROOT = KAGGLE_INPUT
os.environ["IMAGE_ROOT"] = IMAGE_ROOT

# ── Reports (.txt files) ─────────────────────────────────────────────────────
REPORTS_ROOT = find_mount(REPORTS_SLUG)
if not REPORTS_ROOT:
    raise FileNotFoundError(
        f"Dataset '{REPORTS_SLUG}' not attached. Add it via Kaggle: Add Data → Datasets."
    )
os.environ["REPORTS_ROOT"] = REPORTS_ROOT
print(f"REPORTS_ROOT (txt reports):   {REPORTS_ROOT}")

# ── Verify metadata CSVs ─────────────────────────────────────────────────────
for fname in ["mimic-cxr-2.0.0-split.csv",
              "mimic-cxr-2.0.0-chexpert.csv",
              "mimic-cxr-2.0.0-metadata.csv"]:
    path = os.path.join(KAGGLE_INPUT, fname)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing: {path}")
print("All metadata CSVs present.")

# ── Parse helpers (defined at module level so multiprocessing can pickle them) ─
SECTION_PAT = re.compile(
    r"(FINAL REPORT|EXAMINATION|INDICATION|TECHNIQUE|COMPARISON|HISTORY|"
    r"FINDINGS|IMPRESSION|RECOMMENDATION|NOTIFICATION|CLINICAL HISTORY|REASON FOR EXAMINATION|"
    r"WET READ|WET READ VERSION):",
    re.IGNORECASE,
)

def extract_findings(text):
    parts = SECTION_PAT.split(text)
    sections = {}
    for i in range(1, len(parts), 2):
        sections[parts[i].upper().strip()] = parts[i + 1].strip()
    return sections.get("FINDINGS") or sections.get("IMPRESSION") or ""

def parse_one_report(txt_path):
    try:
        study_id = int(os.path.basename(txt_path)[1:-4])
        with open(txt_path, encoding="utf-8") as f:
            findings = extract_findings(f.read()).replace("\n", " ").strip()
    except (ValueError, OSError):
        return None
    if not findings:
        return None
    return (study_id, findings, os.path.basename(txt_path))

# ── Step 1: Collect all .txt report files ────────────────────────────────────
REPORTS_LOCAL = "/kaggle/working/mimic_cxr_cleaned.csv"

txt_files = glob.glob(f"{REPORTS_ROOT}/files/**/s*.txt", recursive=True)
print(f"Found {len(txt_files)} report .txt files")
if len(txt_files) == 0:
    raise FileNotFoundError(
        f"No .txt report files found under {REPORTS_ROOT}/files/. "
        "Check that mimic-cxr-reported dataset has a files/ directory."
    )

# ── Step 2: Parse reports in parallel (CPU workers, not GPU) ─────────────────
n_workers = max(1, cpu_count())
print(f"Parsing with {n_workers} CPU workers (multiprocessing)…")
with Pool(processes=n_workers) as pool:
    results = pool.map(parse_one_report, txt_files, chunksize=500)

rows = [r for r in results if r is not None]
reports = pd.DataFrame(rows, columns=["study_id", "findings", "Note_file"])
print(f"Extracted {len(reports)} reports with non-empty findings")

# ── Step 3: Merge with metadata ───────────────────────────────────────────────
metadata = pd.read_csv(f"{KAGGLE_INPUT}/mimic-cxr-2.0.0-metadata.csv")
df = metadata.merge(reports, on="study_id", how="inner")

# ── Step 4: Vectorized path construction (no row-by-row apply) ───────────────
subj = df["subject_id"].astype(str)
sid  = df["study_id"].astype(str)
df["Img_Folder"]   = "p" + subj.str[:2] + "/p" + subj + "/s" + sid
df["Img_Filename"] = df["dicom_id"].astype(str) + ".jpg"
df = df[["dicom_id", "findings", "Img_Folder", "Img_Filename", "Note_file"]]

# ── Step 5: Bulk filesystem scan instead of 227k individual stat() calls ─────
print("Scanning IMAGE_ROOT for existing JPGs…")
all_jpgs = set()
for root, _, files in os.walk(IMAGE_ROOT):
    rel = os.path.relpath(root, IMAGE_ROOT)
    for fname in files:
        if fname.endswith(".jpg"):
            all_jpgs.add(f"{rel}/{fname}" if rel != "." else fname)
print(f"Found {len(all_jpgs)} JPGs on disk")

rel_paths = df["Img_Folder"] + "/" + df["Img_Filename"]
df = df[rel_paths.isin(all_jpgs)].reset_index(drop=True)

df.to_csv(REPORTS_LOCAL, index=False)
os.environ["REPORTS_CSV"] = REPORTS_LOCAL
print(f"Built {REPORTS_LOCAL}: {len(df)} rows")
print(f"\nSample:")
print(df[["Img_Folder", "Img_Filename"]].head(3).to_string())


## Cell 5 — Write `configs/env_config.yaml` with Kaggle Paths

In [ ]:
import os
import subprocess

result = subprocess.run(
    "readlink -f $(which java) | sed 's|/bin/java||'",
    shell=True, capture_output=True, text=True
)
java_home = result.stdout.strip() or "/usr/lib/jvm/java-8-openjdk-amd64/jre"
java_path = java_home + "/bin:"

KAGGLE_INPUT = os.environ.get("KAGGLE_INPUT", "/kaggle/input/mimic-cxr-jpg-lite")
IMAGE_ROOT   = os.environ.get("IMAGE_ROOT",   KAGGLE_INPUT)
REPORTS_CSV  = os.environ.get("REPORTS_CSV",  "/kaggle/working/mimic_cxr_cleaned.csv")

env_config_content = f"""paths:
  data_root: \"{KAGGLE_INPUT}\"
  mimic_cxr_jpg_root: \"{IMAGE_ROOT}\"
  split_csv: \"{KAGGLE_INPUT}/mimic-cxr-2.0.0-split.csv\"
  reports_csv: \"{REPORTS_CSV}\"
  chexpert_csv: \"{KAGGLE_INPUT}/mimic-cxr-2.0.0-chexpert.csv\"
  metadata_csv: \"{KAGGLE_INPUT}/mimic-cxr-2.0.0-metadata.csv\"
  output_dir: \"/kaggle/working/output\"
  checkpoint_dir: \"/kaggle/working/checkpoints\"

wandb:
  entity: \"\"
  project: \"meta-cxr\"

java:
  home: \"{java_home}\"
  path: \"{java_path}\"
"""

os.makedirs("configs", exist_ok=True)
with open("configs/env_config.yaml", "w") as f:
    f.write(env_config_content)

print("Written configs/env_config.yaml:")
print(env_config_content)


## Cell 6 — Launch 2-GPU DDP Training

Uses `torch.distributed.run` (alias for `torchrun`) with `--standalone` for single-node multi-GPU.  
Training output is streamed live. Expect each epoch to take 30–90 minutes depending on dataset size.

In [ ]:
import subprocess
import sys
import os

os.makedirs("/kaggle/working/output", exist_ok=True)
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)

cmd = [
    sys.executable, "-m", "torch.distributed.run",
    "--standalone",
    "--nproc_per_node=2",
    "--master_port=12355",
    "-m", "pretraining.train",
    "--cfg-path", "pretraining/configs/mimic_cxr_2gpu.yaml",
]

print("Launch command:")
print(" ".join(cmd))
print("\n" + "="*60 + "\n")

env = os.environ.copy()
env["PYTHONPATH"] = "/kaggle/working/META-CXR"

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd="/kaggle/working/META-CXR",
    env=env,
)

for line in process.stdout:
    print(line, end="", flush=True)

process.wait()
print(f"\n" + "="*60)
print(f"Training finished with exit code: {process.returncode}")

## Cell 7 — Display Evaluation Results

In [ ]:
import json
import os
import glob
import pandas as pd

OUTPUT_DIR = "/kaggle/working/output"

# ── Training logs ────────────────────────────────────────────────────────────
log_files = sorted(glob.glob(f"{OUTPUT_DIR}/**/log.txt", recursive=True))
print(f"Found {len(log_files)} log file(s)")

for log_file in log_files:
    print(f"\n{'='*60}")
    print(f"Log: {log_file}")
    print('='*60)
    records = []
    with open(log_file) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                print(line)
    if records:
        df = pd.DataFrame(records)
        display(df)

# ── Prediction files ─────────────────────────────────────────────────────────
pred_files = sorted(glob.glob(f"{OUTPUT_DIR}/**/predictions_*.txt", recursive=True))
print(f"\nFound {len(pred_files)} prediction file(s)")

for pred_file in pred_files[:2]:
    print(f"\n{'='*60}")
    print(f"Predictions: {pred_file}")
    print('='*60)
    with open(pred_file) as f:
        for i, line in enumerate(f):
            if i >= 10:
                print(f"  ... ({sum(1 for _ in open(pred_file))} total lines)")
                break
            print(line, end="")

# ── Checkpoint summary ───────────────────────────────────────────────────────
checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/**/checkpoint_*.pth", recursive=True))
print(f"\nSaved checkpoints ({len(checkpoints)}):")
for ckpt in checkpoints:
    size_mb = os.path.getsize(ckpt) / (1024 ** 2)
    print(f"  {ckpt}  ({size_mb:.1f} MB)")

# ── Best checkpoint info ─────────────────────────────────────────────────────
best_ckpts = glob.glob(f"{OUTPUT_DIR}/**/checkpoint_best.pth", recursive=True)
if best_ckpts:
    print(f"\nBest checkpoint: {best_ckpts[0]}")